<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/phase_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import time
import copy

params = {
    "n_ambiant": 4, "deg_P": 3, "n_terms_poly": 20, "seed": 0, "s_ordre": 3,
    "seeds": {1: 3126, 2: 3821, 3: 123, 42: 41726, 43: 972, 44: 80, 11: 291, 12: 4393},
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "n_train": 2000, "levels": 100, "sigma0": 1.5, "sigma_decay": 0.9,
    "n_dict_candidates": 2000, "n_centers_per_level": 200, "n_test": 5000,
    "n_stop": 1e4, "plateau_window": 5, "plateau_tol": 1e-3,
}
params["n_unlabeled"] = 4000 - params["n_train"]
params["lambda_reg"]  = 4 * (params["weights"][3]**(-1)) * (params["n_train"]**(-2))
params["train_center_ratio"] = 0.5 + params["n_train"] / (2 * 4000)

def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree + 1), repeat=d) if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, exp in enumerate(alpha):
                if exp > 0: term *= x[:, j] ** exp
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices) if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_coeff = np.max(np.abs(coeffs))
    if max_coeff > 0:
        normalized_coeffs = coeffs / max_coeff
        def P_normalized(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(normalized_coeffs, indices):
                term = np.ones(x.shape[0])
                for j, exp in enumerate(alpha):
                    if exp > 0:
                        term *= x[:, j] ** exp
                y += c * term
            return y
        return P_normalized
    else:
        return P

P1, indices1, coeffs1 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][1])
P2, indices2, coeffs2 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][2])
P3, indices3, coeffs3 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][3])

P1 = normalize_polynomial(P1, indices1, coeffs1)
P2 = normalize_polynomial(P2, indices2, coeffs2)
P3 = normalize_polynomial(P3, indices3, coeffs3)
def Q(x): return P1(x) * P2(x) * P3(x)

def grad_Q_analytical(X):
    eps = 1e-5; d = X.shape[1]; p1=P1(X); p2=P2(X); p3=P3(X)
    grad = np.zeros_like(X)
    for k in range(d):
        Xp = X.copy(); Xp[:, k] += eps; Xm = X.copy(); Xm[:, k] -= eps
        dp1=(P1(Xp)-P1(Xm))/(2*eps); dp2=(P2(Xp)-P2(Xm))/(2*eps); dp3=(P3(Xp)-P3(Xm))/(2*eps)
        grad[:, k] = dp1*p2*p3 + p1*dp2*p3 + p1*p2*dp3
    return grad

def project_to_Q_zero(X_init, n_steps=150, lr=0.05, tol=1e-4):
    X = X_init.copy()
    for _ in range(n_steps):
        q=Q(X); gq=grad_Q_analytical(X); gq2=2*q[:,None]*gq
        norm=np.linalg.norm(gq2,axis=1,keepdims=True)+1e-12
        X=X-lr*gq2/norm; X=np.clip(X,0.0,1.0)
        if np.abs(Q(X)).max()<tol*0.1: break
    return X, np.abs(Q(X))<tol

def sample_on_Q_zero(n_target, d, tol=1e-4, oversample=4, seed=None):
    rng=np.random.default_rng(seed); collected=[]; n_collected=0
    while n_collected < n_target:
        n_batch=max((n_target-n_collected)*oversample,200)
        X_init=rng.uniform(0,1,(int(n_batch),d)); X_proj,conv=project_to_Q_zero(X_init,tol=tol)
        good=X_proj[conv]
        if len(good)>0: collected.append(good); n_collected+=len(good)
        print(f"  collectés: {n_collected}/{n_target}  (taux: {conv.mean():.1%})",end='\r')
    print(); return np.vstack(collected)[:n_target]

# UTILITAIRES GAUSSIENNES (phase 1)
# ══════════════════════════════════════════════════════════════════════
def _gauss_base(X, centers, sigma):
    """Retourne diff (n,m,d), phi (n,m) — blocs de base."""
    diff = X[:,None,:] - centers[None,:,:]
    phi  = np.exp(-np.sum(diff**2, axis=2) / (2*sigma**2))
    return diff, phi

def gaussian_phi_grad_hess(X, centers, sigma, weights=None):
    d=X.shape[1]; s2=sigma**2; diff,phi=_gauss_base(X,centers,sigma)
    need_grad = weights is None or weights.get(1,0.)!=0 or weights.get(2,0.)!=0
    grad = -(diff/s2)*phi[:,:,None] if need_grad else None
    need_hess = weights is None or weights.get(2,0.)!=0
    if need_hess:
        s4=s2**2
        hess=(np.einsum('nmi,nmj->nmij',diff,diff)/s4-np.eye(d)[None,None,:,:]/s2)*phi[:,:,None,None]
    else: hess=None
    return phi, grad, hess

def gaussian_all_derivatives(X, centers, sigma, weights=None):
    d=X.shape[1]; s2=sigma**2; diff,phi=_gauss_base(X,centers,sigma)
    grad=-(diff/s2)*phi[:,:,None]
    need_hess = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    if need_hess:
        s4=s2**2
        hess=(np.einsum('nmi,nmj->nmij',diff,diff)/s4-np.eye(d)[None,None,:,:]/s2)*phi[:,:,None,None]
    else: hess=None
    need_third = weights is None or weights.get(3,0.)!=0
    if need_third:
        s4=s2**2; s6=s2**3; Id=np.eye(d)
        cubic=np.einsum('nmi,nmj,nmk->nmijk',diff,diff,diff)/s6
        sym=(np.einsum('ij,nmk->nmijk',Id,diff)+np.einsum('ik,nmj->nmijk',Id,diff)+
             np.einsum('jk,nmi->nmijk',Id,diff))/s4
        third=(-cubic+sym)*phi[:,:,None,None,None]
    else: third=None
    return phi, grad, hess, third

def gaussian_features(X, centers, sigma):
    diff=X[:,None,:]-centers[None,:,:]; return np.exp(-np.sum(diff**2,axis=2)/(2*sigma**2))

# ══════════════════════════════════════════════════════════════════════
# UTILITAIRES DÉRIVÉES PARTIELLES (phase 2)
#
# Noyau de base :  h_k(x;c,σ) = -(x_k-c_k)/σ² · g(x;c,σ)
# Normalisé par max|h_k| empirique sur X_all.
#
# Ses dérivées Sobolev (ordre 0,1,2,3) valent les dérivées de g
# aux ordres 1,2,3,4. Pour éviter le tenseur (n,m,d,d,d,d) de
# l'ordre 4, on l'utilise UNIQUEMENT contracté avec les coefficients
# (delta_third) ou sur n_G points (G).
# ══════════════════════════════════════════════════════════════════════

def _gauss_derivs_up_to_3(diff, phi, sigma):
    """Retourne grad_g, hess_g, third_g depuis diff et phi déjà calculés."""
    d=diff.shape[2]; s2=sigma**2; s4=s2**2; s6=s2**3; Id=np.eye(d)
    grad_g  = -(diff/s2)*phi[:,:,None]
    hess_g  = (np.einsum('nmi,nmj->nmij',diff,diff)/s4
               - Id[None,None,:,:]/s2)*phi[:,:,None,None]
    cubic   = np.einsum('nmi,nmj,nmk->nmijk',diff,diff,diff)/s6
    sym     = (np.einsum('ij,nmk->nmijk',Id,diff)+np.einsum('ik,nmj->nmijk',Id,diff)+
               np.einsum('jk,nmi->nmijk',Id,diff))/s4
    third_g = (-cubic+sym)*phi[:,:,None,None,None]
    return grad_g, hess_g, third_g

def _fourth_contracted_with_coeffs(diff, phi, sigma, coeffs_per_dir):
    """
    Calcule DIRECTEMENT delta_third pour la phase 2 :
      delta_third[x,a,b,c] = Σ_{m,k} fourth_g[x,m,k,a,b,c] · coeffs[m,k]
                            = Σ_{m,k} ∂_m∂_k∂_a∂_b∂_c g · coeffs[m,k] (symétrie)

    On développe fourth_g analytiquement et on contracte avec coeffs
    sans jamais stocker le tenseur (n,m,d,d,d,d).

    fourth_g[x,m,i,j,k,l] = (diff_i·diff_j·diff_k·diff_l/σ⁸
                              - (δ_ij·diff_k·diff_l + 5 perm)/σ⁶
                              + (δ_ij·δ_kl + δ_ik·δ_jl + δ_il·δ_jk)/σ⁴) · phi

    coeffs_per_dir : (m, d) — coefficients reshape des k colonnes sélectionnées
                    coeffs_per_dir[m,k] = coefficient du noyau h_k centré en m
    """
    d=diff.shape[2]; s2=sigma**2; s4=s2**2; s6=s2**3; s8=s2**4
    Id=np.eye(d)
    # phi : (n,m), diff : (n,m,d), coeffs_per_dir : (m,d)
    # c_phi[x,m] = phi[x,m] · Σ_k coeffs_per_dir[m,k] = phi · c_sum
    c = coeffs_per_dir                           # (m, d)
    c_sum = c.sum(axis=1)                        # (m,)   Σ_k coeffs[m,k]

    # Terme quartic : diff_i·diff_j·diff_k·diff_l/σ⁸ · phi · coeffs[m,k]
    # Contracté sur (m,k) : Σ_{m,k} diff[x,m,i]·diff[x,m,j]·diff[x,m,l] · (diff[x,m,k]/σ⁸·phi·c[m,k])
    # = einsum('nmi,nmj,nml,nmk,mk,nm->nabc')
    weighted_phi_c = phi * c_sum[None,:]         # (n,m)  using Σ_k trick when k not in diff
    # Actually for quartic we need diff_k contracted with c[m,k]:
    diff_c = np.einsum('nmd,md->nm', diff, c)    # (n,m)  Σ_k diff_k·c[m,k]
    phi_dc = phi * diff_c                         # (n,m)

    quart = np.einsum('nmi,nmj,nml,nm->nijl', diff, diff, diff, phi_dc) / s8  # (n,d,d,d)

    # Terme quadratique 6 termes :
    # δ_ij·diff_k·diff_l : Σ_k diff_k·c[m,k]=diff_c, Σ diff_l → diff_l weighted
    phi_dc2 = phi * diff_c                        # (n,m)  Σ_k diff_k·c_k
    phi_cs  = phi * c_sum[None,:]                 # (n,m)  Σ_k c_k

    # Les 6 paires :
    # (i↔j)·(k·l) : δ_ij · (Σ_k diff_k·c_k) · diff_l → scalar_ij * diff_l
    # (i↔k)·(j·l) : δ_ik · (Σ_k diff_j·... ) — ici k est l'indice contracté
    # etc. On les traite analytiquement:

    # Σ_{m,k} δ_ij·diff_k·diff_l·phi·c[m,k] = δ_ij · (Σ_m diff_c[n,m]·diff[n,m,l]·phi) ...
    # Trop complexe à développer proprement ici.
    # Approche plus simple : boucle sur k (d=4 iterations)
    result = np.zeros((diff.shape[0], d, d, d))  # (n, a, b, c)
    for k in range(d):
        ck = c[:, k]                              # (m,)
        # fourth_g[:,:,k,:,:,:] · ck · phi — shape (n,d,d,d)
        # fourth_g[x,m,k,a,b,c] = (diff_k·diff_a·diff_b·diff_c/s8
        #   - δ_ka·diff_b·diff_c/s6 - δ_kb·diff_a·diff_c/s6 - δ_kc·diff_a·diff_b/s6
        #   - δ_ab·diff_k·diff_c/s6 - δ_ac·diff_k·diff_b/s6 - δ_bc·diff_k·diff_a/s6
        #   + δ_ka·δ_bc/s4 + δ_kb·δ_ac/s4 + δ_kc·δ_ab/s4
        #   + δ_ab·δ_kc/s4 + δ_ac·δ_kb/s4 + δ_bc·δ_ka/s4) · phi
        # Les termes δ_k* valent 1 seulement si l'indice = k
        phi_ck = phi * ck[None,:]                 # (n,m)

        # Terme quartic : diff_k·diff_a·diff_b·diff_c
        dk = diff[:,:,k]                          # (n,m)
        result += np.einsum('nm,nma,nmb,nmc->nabc', phi_ck*dk, diff,diff,diff) / s8

        # Termes quadratiques (6 paires impliquant k) :
        result -= np.einsum('nm,nmb,nmc->nabc', phi_ck*dk,diff,diff)[:,None,:,:] / s6 * np.eye(d)[:,:,None,None] # δ_ka
        # La boucle sur 6 paires est longue — on regroupe :
        for a in range(d):
            # δ_ka : contribution quand a=k
            if a == k:
                result[:,a,:,:] -= np.einsum('nm,nmb,nmc->nbc', phi_ck, diff,diff) / s6
                result[:,a,:,:] += np.einsum('nm,bc->nbc', phi_ck, np.eye(d)) / s4
            for b in range(d):
                if b == k:
                    result[:,a,b,:] -= np.einsum('nm,nma,nmc->nac', phi_ck, diff,diff)[:,a,:] / s6
                    result[:,a,b,:] += np.einsum('nm->n', phi_ck)[:,None] * np.eye(d)[a,:] / s4
                for c_idx in range(d):
                    if c_idx == k:
                        result[:,a,b,c_idx] -= np.einsum('nm,nma,nmb->n', phi_ck, diff,diff) / s6
                    # δ_ab
                    if a == b:
                        result[:,a,b,c_idx] -= np.einsum('nm,nmd->n', phi_ck*dk, diff)[:,] / s6 * (c_idx == c_idx)
                    # δ_ac
                    if a == c_idx:
                        result[:,a,b,c_idx] -= np.einsum('nm,nmb->n', phi_ck*dk, diff) / s6
                    # δ_bc
                    if b == c_idx:
                        result[:,a,b,c_idx] -= np.einsum('nm,nma->n', phi_ck*dk, diff) / s6
                    # termes constants δδ
                    if a==b: result[:,a,b,c_idx] += np.einsum('nm->n', phi_ck) / s4 * (c_idx == k)
                    if a==c_idx: result[:,a,b,c_idx] += np.einsum('nm->n', phi_ck) / s4 * (b == k)
                    if b==c_idx: result[:,a,b,c_idx] += np.einsum('nm->n', phi_ck) / s4 * (a == k)

    return result

# Abandon de l'approche boucle — trop de risques d'erreurs.
# Approche correcte : calculer fourth_g sur n_G points seulement (500 max → 205 MB)

def _fourth_g_on_subset(diff_sub, phi_sub, sigma):
    """
    Calcule fourth_g sur un sous-ensemble de points (n_sub, m, d, d, d, d).
    Avec n_sub=500, m=200, d=4 : 205 MB — acceptable.
    """
    d=diff_sub.shape[2]; s2=sigma**2; s4=s2**2; s6=s2**3; s8=s2**4; Id=np.eye(d)
    quart = np.einsum('nmi,nmj,nmk,nml->nmijkl',diff_sub,diff_sub,diff_sub,diff_sub)/s8
    quad  = -(
        np.einsum('ij,nmk,nml->nmijkl',Id,diff_sub,diff_sub)+
        np.einsum('ik,nmj,nml->nmijkl',Id,diff_sub,diff_sub)+
        np.einsum('il,nmj,nmk->nmijkl',Id,diff_sub,diff_sub)+
        np.einsum('jk,nmi,nml->nmijkl',Id,diff_sub,diff_sub)+
        np.einsum('jl,nmi,nmk->nmijkl',Id,diff_sub,diff_sub)+
        np.einsum('kl,nmi,nmj->nmijkl',Id,diff_sub,diff_sub)
    )/s6
    const = (np.einsum('ij,kl->ijkl',Id,Id)+
             np.einsum('ik,jl->ijkl',Id,Id)+
             np.einsum('il,jk->ijkl',Id,Id))[None,None,:,:,:,:]/s4
    return (quart+quad+const)*phi_sub[:,:,None,None,None,None]

def _partial_gauss_features_and_norms(X, centers, sigma, X_for_norm):
    """
    Features h_k normalisées pour la phase 2.
    Retourne A (n, m*d) et norms (m, d).
    """
    d=X.shape[1]; s2=sigma**2
    diff_X,phi_X    = _gauss_base(X, centers, sigma)
    diff_n,phi_n    = _gauss_base(X_for_norm, centers, sigma)
    h_n  = -(diff_n/s2)*phi_n[:,:,None]          # (N_all,m,d)
    norms= np.maximum(np.abs(h_n).max(axis=0), 1e-12)  # (m,d)
    h_X  = -(diff_X/s2)*phi_X[:,:,None]          # (n,m,d)
    return (h_X/norms[None,:,:]).reshape(X.shape[0],-1), norms

def _partial_gauss_sobolev_score(X_score, candidates, sigma, weights):
    """
    Phase score pour le dictionnaire (phase 2).
    Calculé sur X_score (n_G points) et SANS hess_h pour éviter les 4 GB.
    Le score approximé phi+grad suffit pour classer les candidats.
    """
    d=X_score.shape[1]; n=X_score.shape[0]; m=candidates.shape[0]; s2=sigma**2
    diff,phi = _gauss_base(X_score, candidates, sigma)
    h = -(diff/s2)*phi[:,:,None]
    norms = np.maximum(np.abs(h).max(axis=0), 1e-12)   # (m, d)
    phi_h = (h/norms[None,:,:]).reshape(n, m*d)
    need_grad = weights is None or weights.get(1,0.)!=0
    if need_grad:
        s4=s2**2
        hess_g=(np.einsum('nmi,nmj->nmij',diff,diff)/s4
                -np.eye(d)[None,None,:,:]/s2)*phi[:,:,None,None]
        grad_h=(hess_g/norms[None,:,:,None]).reshape(n,m*d,d)
    else: grad_h=None
    return phi_h, grad_h, None   # hess_h=None : trop coûteux sur le dictionnaire complet

def _partial_gauss_full(X_sub, sel_centers, sel_dirs, sigma, norms_sel, weights):
    """
    Calcule phi_h, grad_h, hess_h, third_h pour les k noyaux SÉLECTIONNÉS
    sur un sous-ensemble X_sub (train, test, ou n_G points).

    sel_centers : (k, d) — centres sélectionnés
    sel_dirs    : (k,)   — direction k de chaque noyau
    norms_sel   : (k,)   — facteurs de normalisation
    """
    n=X_sub.shape[0]; k=len(sel_dirs); d=X_sub.shape[1]; s2=sigma**2; s4=s2**2
    diff,phi = _gauss_base(X_sub, sel_centers, sigma)
    # phi_h[x,j] = h_{dir[j]}(x;c_j) / norm_j
    h = -(diff/s2)*phi[:,:,None]                  # (n,k,d)
    phi_h = h[np.arange(n)[:,None], np.arange(k)[None,:], sel_dirs[None,:]] / norms_sel[None,:]
    # = -(diff[:,:,sel_dirs]/s2)*phi / norms_sel
    phi_h = -(diff[np.arange(n)[:,None],np.arange(k)[None,:],sel_dirs[None,:]]/s2
               )*phi / norms_sel[None,:]           # (n,k)

    _,hess_g,third_g = _gauss_derivs_up_to_3(diff, phi, sigma)
    # grad_h[x,j,a] = third_g[x,j,sel_dirs[j],a] ... wait:
    # ∂_a h_{dk} = hess_g[x,j,dk,a]
    # grad_h[x,j,a] = hess_g[x,j,sel_dirs[j],a] / norms_sel[j]
    need_grad = weights is None or weights.get(1,0.)!=0 or weights.get(2,0.)!=0
    if need_grad:
        grad_h = hess_g[np.arange(n)[:,None], np.arange(k)[None,:],
                        sel_dirs[None,:], :] / norms_sel[None,:,None]  # (n,k,d)
    else: grad_h=None

    need_hess = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    if need_hess:
        # hess_h[x,j,a,b] = third_g[x,j,sel_dirs[j],a,b] / norms_sel[j]
        hess_h = third_g[np.arange(n)[:,None], np.arange(k)[None,:],
                         sel_dirs[None,:], :, :] / norms_sel[None,:,None,None]  # (n,k,d,d)
    else: hess_h=None

    need_third = weights is None or weights.get(3,0.)!=0
    if need_third:
        # third_h[x,j,a,b,c] = fourth_g[x,j,sel_dirs[j],a,b,c] / norms_sel[j]
        # On calcule fourth_g sur X_sub seulement
        fourth_g = _fourth_g_on_subset(diff, phi, sigma)   # (n,k,d,d,d,d)
        third_h  = fourth_g[np.arange(n)[:,None], np.arange(k)[None,:],
                             sel_dirs[None,:], :, :, :] / norms_sel[None,:,None,None,None]
        # (n,k,d,d,d)
    else: third_h=None

    return phi_h, grad_h, hess_h, third_h

# ══════════════════════════════════════════════════════════════════════
# FONCTIONS COMMUNES
# ══════════════════════════════════════════════════════════════════════
def sobolev_norms_sq_partial(phi, grad, hess, weights, n):
    norms=np.zeros(phi.shape[1])
    w0=weights.get(0,0.)
    if w0: norms+=w0*np.einsum('xj,xj->j',phi,phi)/n
    w1=weights.get(1,0.)
    if w1 and grad is not None: norms+=w1*np.einsum('xjk,xjk->j',grad,grad)/n
    w2=weights.get(2,0.)
    if w2 and hess is not None: norms+=w2*np.einsum('xjkl,xjkl->j',hess,hess)/n
    return norms

def sobolev_norms_sq(phi, grad, hess, third, weights, n):
    norms=sobolev_norms_sq_partial(phi,grad,hess,weights,n)
    w3=weights.get(3,0.)
    if w3 and third is not None: norms+=w3*np.einsum('xjklm,xjklm->j',third,third)/n
    return norms

def build_G(phi, grad, hess, third, weights, n):
    G=np.zeros((phi.shape[1],phi.shape[1]))
    w0=weights.get(0,0.)
    if w0: G+=w0*(phi.T@phi)/n
    w1=weights.get(1,0.)
    if w1 and grad  is not None: G+=w1*np.einsum('xik,xjk->ij',    grad, grad, optimize='optimal')/n
    w2=weights.get(2,0.)
    if w2 and hess  is not None: G+=w2*np.einsum('xikl,xjkl->ij',  hess, hess, optimize='optimal')/n
    w3=weights.get(3,0.)
    if w3 and third is not None: G+=w3*np.einsum('xiklm,xjklm->ij',third,third,optimize='optimal')/n
    return G

def sample_candidates(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr=min(int(round(n_candidates*train_ratio)),X_train.shape[0])
    n_ul=min(n_candidates-n_tr,X_unlabeled.shape[0]); parts=[]
    if n_tr>0: parts.append(X_train[rng.choice(X_train.shape[0],n_tr,replace=False)])
    if n_ul>0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0],n_ul,replace=False)])
    return np.vstack(parts)

# ══════════════════════════════════════════════════════════════════════
print("Génération du cloud sur {Q=0}...")
d = params["n_ambiant"]
X_train     = sample_on_Q_zero(params["n_train"],     d, seed=params["seeds"][42])
X_test      = sample_on_Q_zero(params["n_test"],      d, seed=params["seeds"][43])
X_unlabeled = sample_on_Q_zero(params["n_unlabeled"], d, seed=params["seeds"][44])
print(f"  X_train:{X_train.shape}  X_test:{X_test.shape}  X_unlabeled:{X_unlabeled.shape}")

P_target1, indicest1, coeffst1 =  random_sparse_polynomial(params["n_ambiant"],
                                           7, params["n_terms_poly"], seed= 11)
#def P_target1_normalized(X):
 #   return P_target1(X) / ((P_target1(_X0)**2 + 1)**0.5)
#P_target2, _, _ = random_sparse_polynomial(params["n_ambiant"],
#                                           7, params["n_terms_poly"], seed= 12)
Ptarget1 = normalize_polynomial(P_target1, indicest1, coeffst1)

def target_function(X):
    return  5  * np.sin (P_target1(3*X))
#X[:,3]**2+3+20*np.maximum(np.sin(X[:,0]*X[:,1]**3-8*X[:,2]**3+7*X[:,3]**9),0)

y_train = target_function(X_train); y_test = target_function(X_test)
X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)

# ══════════════════════════════════════════════════════════════════════
# BOUCLE GÉNÉRIQUE
# ══════════════════════════════════════════════════════════════════════
def run_phase(phase, pred_train, pred_test, fi_derivs, history, losses, times, level_offset=0):
    pred_train=pred_train.copy(); pred_test=pred_test.copy()
    fi_derivs=copy.deepcopy(fi_derivs); history=list(history)
    losses=list(losses); times=list(times)
    stop_reason=None; final_level=None

    print(f"\n{'='*70}")
    print(f"PHASE {phase} : {'Gaussiennes' if phase==1 else '∂ₖ Gaussiennes normalisées'}")
    print(f"{'='*70}")

    for level in range(params["levels"]):
        abs_level = level+level_offset
        t0=time.time()
        sigma=params["sigma0"]*params["sigma_decay"]**level
        lambda_reg=params["lambda_reg"]*(sigma/params["sigma0"])**2
        lambda_reg=max(lambda_reg,1e-7)
        rng=np.random.default_rng(seed=abs_level)

        candidates=sample_candidates(X_train,X_unlabeled,
                                     params["n_dict_candidates"],
                                     params["train_center_ratio"],rng)
        residual=y_train-pred_train

        # ── Score dictionnaire ──────────────────────────────────────────
        if phase==1:
            phi_c,grad_c,hess_c = gaussian_phi_grad_hess(X_all,candidates,sigma,params["weights"])
            A_cand = gaussian_features(X_train,candidates,sigma)
        else:
            # Score sur n_G points seulement (évite 4 GB pour hess sur dictionnaire complet)
            n_G_score = min(N_all, 500)
            idx_score = rng.integers(0, N_all, size=n_G_score)
            phi_c,grad_c,hess_c = _partial_gauss_sobolev_score(
                X_all[idx_score],candidates,sigma,params["weights"])
            A_cand,_ = _partial_gauss_features_and_norms(X_train,candidates,sigma,X_all)

        corr     = (A_cand.T@residual)/len(X_train)
        norms_sq = sobolev_norms_sq_partial(phi_c,grad_c,hess_c,params["weights"],N_all)
        norms_sq = np.maximum(norms_sq,1e-12)
        scores   = corr**2/norms_sq

        k=min(params["n_centers_per_level"],len(candidates))
        top_k=np.argsort(scores)[-k:]

        # ── Features & dérivées sur les centres sélectionnés ───────────
        if phase==1:
            centers=candidates[top_k]
            A     =gaussian_features(X_train,centers,sigma)
            Atest =gaussian_features(X_test, centers,sigma)
            phi,grad,hess,third=gaussian_all_derivatives(X_all,centers,sigma,params["weights"])

            n_G=int(min(N_all,max(500,int(-params["n_train"]+8*sigma**(-3)))))
            n_G=min(n_G,2000); idx_G=rng.choice(N_all,size=n_G,replace=False)
            phi_G=phi[idx_G]
            grad_G =grad[idx_G]  if grad  is not None else None
            hess_G =hess[idx_G]  if hess  is not None else None
            third_G=third[idx_G] if third is not None else None

        else:
            # Phase 2 : dictionnaire étendu m*d, on sélectionne dans top_k
            sel_center_idx = top_k // d     # indice du centre dans candidates
            sel_dirs       = top_k % d      # direction de la dérivée partielle
            sel_centers    = candidates[sel_center_idx]   # (k, d)

            # ── Normalisation : un seul passage sur X_all[idx_score] ──────────
            # (les mêmes 500 pts utilisés pour le score)
            diff_sc, phi_sc = _gauss_base(X_all[idx_score], candidates, sigma)
            h_sc = -(diff_sc/sigma**2)*phi_sc[:,:,None]   # (n_G_score, m_cand, d)
            norms_all = np.maximum(np.abs(h_sc).max(axis=0), 1e-12)  # (m_cand, d)
            norms_sel = norms_all[sel_center_idx, sel_dirs]           # (k,)

            # ── Features train (un seul passage) ─────────────────────────────
            diff_tr, phi_tr = _gauss_base(X_train, sel_centers, sigma)
            h_tr = -(diff_tr/sigma**2)*phi_tr[:,:,None]               # (n_tr, k, d)
            _k = np.arange(k)
            A   = (h_tr[:, _k, sel_dirs] / norms_sel[None,:]) # (n_tr, k) — colonnes dir_j

            # ── Features test (un seul passage) ──────────────────────────────
            diff_te, phi_te = _gauss_base(X_test, sel_centers, sigma)
            h_te = -(diff_te/sigma**2)*phi_te[:,:,None]
            Atest = h_te[:, _k, sel_dirs] / norms_sel[None,:]         # (n_te, k)

            # ── Dérivées Sobolev sur n_G pts (pour G et fi_derivs) ───────────
            n_G = int(min(N_all, max(500, int(-params["n_train"]+8*sigma**(-3)))))
            n_G = min(n_G, 500)
            idx_G = rng.choice(N_all, size=n_G, replace=False)
            phi_G,grad_G,hess_G,third_G = _partial_gauss_full(
                X_all[idx_G], sel_centers, sel_dirs, sigma, norms_sel, params["weights"])

            # ── Dérivées sur X_all (phi,grad,hess SEULEMENT — pas third) ─────
            # third sur N_all est trop lourd; on l'approxime sur n_G (cohérent avec G)
            phi,grad,hess,_ = _partial_gauss_full(
                X_all, sel_centers, sel_dirs, sigma, norms_sel,
                {0:params["weights"].get(0,0.), 1:params["weights"].get(1,0.),
                 2:params["weights"].get(2,0.), 3:0.})  # third=None sur N_all
            # Pour delta_third on utilise les n_G points
            third = third_G   # approximation sur n_G pts pour fi_derivs

        G   = build_G(phi_G,grad_G,hess_G,third_G,params["weights"],n_G)
        n   = A.shape[0]; M=(A.T@A)/n+lambda_reg*G; rhs=(A.T@residual)/n

        print(f"\n=== P{phase} L{level} (abs={abs_level}) | σ={sigma:.4f} | "
              f"λ={lambda_reg:.2e} | k={k} | n_G={n_G} | "
              f"score max={scores[top_k[-1]]:.3e} ===")

        if fi_derivs is not None:
            b=np.zeros(k)
            w0=params["weights"].get(0,0.)
            if w0: b+=w0*(fi_derivs['phi']@phi)/N_all
            w1=params["weights"].get(1,0.)
            if w1 and grad is not None:
                b+=w1*np.einsum('xk,xjk->j',fi_derivs['grad'],grad,optimize='optimal')/N_all
            w2=params["weights"].get(2,0.)
            if w2 and hess is not None:
                b+=w2*np.einsum('xkl,xjkl->j',fi_derivs['hess'],hess,optimize='optimal')/N_all
            w3=params["weights"].get(3,0.)
            if w3 and third is not None:
                # phase 2 : third sur n_G pts → utiliser idx_G pour fi_derivs aussi
                _idx = idx_G if phase==2 else np.arange(N_all)
                _n   = n_G  if phase==2 else N_all
                b+=w3*np.einsum('xklm,xjklm->j',
                                fi_derivs['third'][_idx],third,optimize='optimal')/_n
            rhs=rhs-lambda_reg*b

        eigvals,eigvecs=np.linalg.eigh(M); thresh=max(lambda_reg,1e-7); mask=eigvals>thresh
        V=eigvecs[:,mask]; S=eigvals[mask]; coeffs=V@((V.T@rhs)/S)

        loss_data=np.mean((residual-A@coeffs)**2)
        loss_reg =lambda_reg*coeffs@G@coeffs
        loss_total=loss_data+loss_reg
        t1=time.time(); times.append(t1-t0)
        print(f"  rang={mask.sum()}/{k}  |coeff|_max={np.max(np.abs(coeffs)):.4f}  temps={times[-1]:.1f}s")
        print(f"  loss={loss_total:.6f}  (data={loss_data:.6f}  reg={loss_reg:.6f})")

        # Arrêt explosion
        if len(history)>0:
            prev_loss=history[-1]['loss_total']
            if loss_total>prev_loss+params["n_stop"]*lambda_reg:
                best_idx=len(history)-1
                for i in range(len(history)-2,-1,-1):
                    if history[i]['loss_total']<=history[i+1]['loss_total']: best_idx=i
                    else: break
                print(f"  ⚠ ARRÊT explosion (phase {phase}) → niveau {history[best_idx]['level']+1}")
                stop_reason="explosion"; final_level=best_idx; break

        # Mise à jour
        pred_train_new=pred_train+A@coeffs; pred_test_new=pred_test+Atest@coeffs
        delta_phi  =phi@coeffs
        delta_grad =np.einsum('xjk,j->xk',   grad, coeffs) if grad  is not None else None
        delta_hess =np.einsum('xjkl,j->xkl', hess, coeffs) if hess  is not None else None
        # En phase 2, third est sur n_G pts — on le recalcule sur N_all pour fi_derivs
        if phase==2 and params["weights"].get(3,0.)!=0:
            _,_,_,third_full_all = _partial_gauss_full(
                X_all, sel_centers, sel_dirs, sigma, norms_sel,
                {0:0.,1:0.,2:0.,3:params["weights"].get(3,0.)})
            delta_third = np.einsum('xjklm,j->xklm',third_full_all,coeffs)
        else:
            delta_third=np.einsum('xjklm,j->xklm',third,coeffs) if third is not None else None

        def _add(a,b): return a+b if (a is not None and b is not None) else None
        if fi_derivs is None:
            fi_derivs_new={'phi':delta_phi,'grad':delta_grad,'hess':delta_hess,'third':delta_third}
        else:
            fi_derivs_new={'phi':fi_derivs['phi']+delta_phi,
                           'grad':_add(fi_derivs['grad'],delta_grad),
                           'hess':_add(fi_derivs['hess'],delta_hess),
                           'third':_add(fi_derivs['third'],delta_third)}

        mse=mean_squared_error(y_test,pred_test_new); losses.append(mse)
        print(f"  MSE level {abs_level+1} = {mse:.6f}")
        pred_train=pred_train_new; pred_test=pred_test_new; fi_derivs=fi_derivs_new
        history.append({'level':abs_level,'pred_train':pred_train.copy(),
                        'pred_test':pred_test.copy(),'fi_derivs':copy.deepcopy(fi_derivs),
                        'mse':mse,'loss_total':loss_total})

        # Arrêt plateau
        w=params["plateau_window"]
        if len(history)>=w:
            recent=[h['loss_total'] for h in history[-w:]]
            if recent[0]-recent[-1]<params["plateau_tol"]:
                print(f"  ⚠ ARRÊT plateau (phase {phase})")
                stop_reason="plateau"; final_level=len(history)-2; break

    if stop_reason is None: final_level=len(history)-1; stop_reason="max_levels"
    return losses,times,pred_train,pred_test,fi_derivs,history,stop_reason,final_level

# ══════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════
losses=[]; times=[]
pred_train=np.zeros(len(X_train)); pred_test=np.zeros(len(X_test))
fi_derivs=None; history=[]

losses,times,pred_train,pred_test,fi_derivs,history,stop1,final1 = run_phase(
    1,pred_train,pred_test,fi_derivs,history,losses,times,level_offset=0)
n_p1=len(history)
print(f"\n→ Phase 1 ({stop1}). MSE : {history[final1]['mse']:.6f}")

losses,times,pred_train,pred_test,fi_derivs,history,stop2,final2 = run_phase(
    2,pred_train,pred_test,fi_derivs,history,losses,times,level_offset=n_p1)

mse_p1=history[final1]['mse']; mse_p2=history[final2]['mse']
mse_final=min(mse_p1,mse_p2); best_phase=1 if mse_p1<=mse_p2 else 2
print(f"\n{'='*70}")
print(f"Phase 1 : {mse_p1:.6f} ({stop1})  |  Phase 2 : {mse_p2:.6f} ({stop2})")
print(f"→ Meilleure phase : {best_phase}  MSE finale : {mse_final:.6f}")

from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

print("\nCalibration RBF...")
gamma_grid=np.logspace(-3,2,20)
param_grid={'alpha':[1e-8,1e-5,1e-3],'gamma':gamma_grid,'kernel':['rbf']}
kr_cv=GridSearchCV(KernelRidge(),param_grid=param_grid,cv=5,
                   scoring='neg_mean_squared_error',n_jobs=-1)
kr_cv.fit(X_train,y_train)
best_gamma=kr_cv.best_params_['gamma']; best_alpha=kr_cv.best_params_['alpha']
kr_best=KernelRidge(kernel='rbf',gamma=best_gamma,alpha=best_alpha)
kr_best.fit(X_train,y_train)
best_err_rbf=mean_squared_error(y_test,kr_best.predict(X_test))
results=kr_cv.cv_results_; order=np.argsort(results['rank_test_score'])
print("Top 3 RBF :")
for i in range(3):
    idx=order[i]; g=results['param_gamma'][idx]; a=results['param_alpha'][idx]
    kr=KernelRidge(kernel='rbf',gamma=float(g),alpha=float(a)); kr.fit(X_train,y_train)
    print(f"  #{i+1} γ={g:.4f} α={a:.0e} CV={-results['mean_test_score'][idx]:.4f} TEST={mean_squared_error(y_test,kr.predict(X_test)):.6f}")

poly=PolynomialFeatures(degree=min(params["deg_P"],8),include_bias=False)
ridge_poly=Ridge(alpha=1e-8); ridge_poly.fit(poly.fit_transform(X_train),y_train)
err_poly=mean_squared_error(y_test,ridge_poly.predict(poly.transform(X_test)))

print(f"\n{'='*70}\nRÉSULTATS\n{'='*70}")
print(f"Polynomial Ridge : {err_poly:.6f}")
print(f"RBF best         : {best_err_rbf:.6f}  (γ={best_gamma:.4f} α={best_alpha:.0e})")
print(f"Sobolev MS P{best_phase}   : {mse_final:.6f}")
print("\nDécroissance :")
for i,loss in enumerate(losses):
    p="P1" if i<n_p1 else "P2"
    f1="←" if i==final1 else ""; f2="←" if i==final2 else ""
    print(f"  [{p}] {i+1:3d}: {loss:.6f}  ({times[i]:.1f}s) {f1}{f2}")

Génération du cloud sur {Q=0}...
  collectés: 4127/2000  (taux: 51.6%)
  collectés: 10314/5000  (taux: 51.6%)
  collectés: 4196/2000  (taux: 52.4%)
  X_train:(2000, 4)  X_test:(5000, 4)  X_unlabeled:(2000, 4)

PHASE 1 : Gaussiennes

=== P1 L0 (abs=0) | σ=1.5000 | λ=1.00e-04 | k=300 | n_G=500 | score max=1.730e+01 ===
  rang=4/300  |coeff|_max=1.1921  temps=8.2s
  loss=5.231892  (data=5.229267  reg=0.002625)
  MSE level 1 = 5.257965

=== P1 L1 (abs=1) | σ=1.3500 | λ=8.10e-05 | k=300 | n_G=500 | score max=5.742e-05 ===
  rang=6/300  |coeff|_max=5.1707  temps=8.4s
  loss=4.937755  (data=4.933307  reg=0.004449)
  MSE level 2 = 5.100760

=== P1 L2 (abs=2) | σ=1.2150 | λ=6.56e-05 | k=300 | n_G=500 | score max=3.006e-05 ===
  rang=6/300  |coeff|_max=0.4741  temps=8.7s
  loss=4.932440  (data=4.932434  reg=0.000006)
  MSE level 3 = 5.096801

=== P1 L3 (abs=3) | σ=1.0935 | λ=5.31e-05 | k=300 | n_G=500 | score max=2.165e-05 ===
  rang=6/300  |coeff|_max=0.3667  temps=7.5s
  loss=4.932264  (data=4